In [1]:
import os
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 6g --conf spark.driver.extraJavaOptions=\"-XX:+UseG1GC -XX:+UseStringDeduplication\" pyspark-shell"

%load_ext jsoniqmagic

[Info] Using RumbleDB jar file at: file:///home/shirel/.local/lib/python3.12/site-packages/jsoniq/jars/rumbledb-2.1.7.jar


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/11 21:37:05 WARN Utils: Your hostname, DESKTOP-IA393V7, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/11 21:37:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/06/11 21:37:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [2]:
%%jsoniq
count(json-file("git-archive-huge.json.gz"))

28506909


In [2]:
%%jsoniq
for $e in json-file("git-archive-huge.json.gz")
group by $type := $e.type
order by count($e) descending
return {
    "type": $type,
    "count": count($e)
}

The query output 14 items, which is too many to display. Displaying the first 10 items:
{
  "type": "PushEvent",
  "count": 14271557
}
{
  "type": "CreateEvent",
  "count": 3328235
}
{
  "type": "IssueCommentEvent",
  "count": 2782424
}
{
  "type": "WatchEvent",
  "count": 2552211
}
{
  "type": "IssuesEvent",
  "count": 1463914
}
{
  "type": "PullRequestEvent",
  "count": 1393766
}
{
  "type": "ForkEvent",
  "count": 971107
}
{
  "type": "DeleteEvent",
  "count": 548218
}
{
  "type": "PullRequestReviewCommentEvent",
  "count": 441408
}
{
  "type": "GollumEvent",
  "count": 294042
}


In [4]:
%%jsoniq
for $e in json-file("git-archive-huge.json.gz")
where $e.type eq "PushEvent"
count $pos
where $pos le 5
return {
    "actor_login": $e.actor.login,
    "repo_name": $e.repo.name,
    "created_at": $e.created_at
}

{
  "actor_login": "davidcarlsonberg",
  "repo_name": "PubWlkr/PubWlkr",
  "created_at": "2015-02-20T01:00:01Z"
}
{
  "actor_login": "loomchild",
  "repo_name": "loomchild/reload",
  "created_at": "2015-02-20T01:00:01Z"
}
{
  "actor_login": "lsqshr",
  "repo_name": "lsqshr/nipype",
  "created_at": "2015-02-20T01:00:01Z"
}
{
  "actor_login": "PhancyCat",
  "repo_name": "PhancyCat/HTMLClock",
  "created_at": "2015-02-20T01:00:01Z"
}
{
  "actor_login": "saramartinez",
  "repo_name": "saramartinez/tv-or-not-tv",
  "created_at": "2015-02-20T01:00:01Z"
}


In [5]:
%%jsoniq
count(
    for $e in json-file("git-archive-huge.json.gz")
    where $e.type eq "PushEvent"
    where count($e.payload.commits[]) > 0
    return $e
)

14173075


In [2]:
%%jsoniq
{|
    for $e in json-file("git-archive-huge.json.gz")
    let $date := $e.created_at
    group by $type := $e.type
    return {
        $type: {
            "earliest": min($date),
            "latest": max($date)
        }
    }
|}

{
  "IssueCommentEvent": {
    "earliest": "2015-01-01T00:00:06Z",
    "latest": "2015-02-28T23:59:59Z"
  },
  "CreateEvent": {
    "earliest": "2015-01-01T00:00:01Z",
    "latest": "2015-02-28T23:59:59Z"
  },
  "WatchEvent": {
    "earliest": "2015-01-01T00:00:18Z",
    "latest": "2015-02-28T23:59:58Z"
  },
  "IssuesEvent": {
    "earliest": "2015-01-01T00:00:30Z",
    "latest": "2015-02-28T23:59:59Z"
  },
  "PublicEvent": {
    "earliest": "2015-01-01T00:09:13Z",
    "latest": "2015-02-28T23:57:28Z"
  },
  "PullRequestReviewCommentEvent": {
    "earliest": "2015-01-01T00:00:08Z",
    "latest": "2015-02-28T23:59:26Z"
  },
  "ForkEvent": {
    "earliest": "2015-01-01T00:00:16Z",
    "latest": "2015-02-28T23:59:34Z"
  },
  "ReleaseEvent": {
    "earliest": "2015-01-01T00:02:19Z",
    "latest": "2015-02-28T23:59:06Z"
  },
  "GollumEvent": {
    "earliest": "2015-01-01T00:01:10Z",
    "latest": "2015-02-28T23:59:44Z"
  },
  "DeleteEvent": {
    "earliest": "2015-01-01T00:00:30Z",
    "lat